# GCS Dataset Explorer — Recursive Folder Audit

Notebook này dùng cho **bước 1: khám phá cấu trúc dữ liệu trên Google Cloud Storage**.

Notebook sẽ:
- Quét đệ quy toàn bộ object dưới một `gs://bucket/prefix`.
- Chỉ lấy metadata/path, **không tải ảnh/video xuống**.
- Tạo bảng Pandas theo folder, extension, depth và L21–L30.
- Tìm sample path và keyword như `caption`, `ocr`, `object`, `keyframe`.
- Xuất CSV và Excel để kiểm soát và gửi lại cho ChatGPT.

Sau khi xem cấu trúc thật, bước tiếp theo mới viết summary:
**videos / frames / caption / OCR / objects / missing data / GCS ↔ Elasticsearch**.

## 1. Cài thư viện

In [1]:
%pip install -q google-cloud-storage pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


## 2. GCS Credentials

Notebook này được cấu hình để dùng trực tiếp **Google Cloud Service Account JSON**.

Bạn chỉ cần truyền đường dẫn file secret JSON vào biến:

```python
GCS_CREDENTIALS_PATH = "/path/to/your-service-account.json"
```

Ví dụ trên Google Colab:

```python
GCS_CREDENTIALS_PATH = "/content/gcs-secret.json"
```

Ví dụ trên máy local/Linux:

```python
GCS_CREDENTIALS_PATH = "/home/user/secrets/gcs-service-account.json"
```

> Không commit file credential JSON lên GitHub.


In [ ]:
# Không cần chạy gcloud auth.
# Notebook sẽ đọc Service Account JSON trực tiếp từ GCS_CREDENTIALS_PATH.

## 3. Cấu hình GCS

Bạn chỉ cần sửa 3 biến:

```python
BUCKET_NAME = "your-bucket-name"
ROOT_PREFIX = "your/prefix/"
GCS_CREDENTIALS_PATH = "/path/to/secret.json"
```

Ví dụ:

```python
BUCKET_NAME = "aic-data"
ROOT_PREFIX = "keyframes/"
GCS_CREDENTIALS_PATH = "/content/gcs-service-account.json"
```

Nếu muốn scan toàn bộ bucket:

```python
ROOT_PREFIX = ""
```

`MAX_OBJECTS = None` nghĩa là quét toàn bộ object trong prefix.


In [1]:
# ============================================================
# REQUIRED CONFIG
# ============================================================

BUCKET_NAME = "aic_ai_2026"

# Ví dụ:
# ROOT_PREFIX = "keyframes/"
# ROOT_PREFIX = "annotations/"
# ROOT_PREFIX = ""   # scan toàn bucket
ROOT_PREFIX = ""

# Đường dẫn tới file Google Cloud Service Account JSON
# Ví dụ Colab: "/content/gcs-service-account.json"
# Ví dụ Linux: "/home/user/secrets/gcs-service-account.json"
from pathlib import Path

DEFAULT_GCS_CREDENTIALS_REL = Path("apps/secrets/gen-lang-client-0547522732-410672fac05f.json")

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / DEFAULT_GCS_CREDENTIALS_REL).exists() or (candidate / ".env").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
GCS_CREDENTIALS_PATH = str((REPO_ROOT / DEFAULT_GCS_CREDENTIALS_REL).resolve())


# ============================================================
# OPTIONAL CONFIG
# ============================================================

MAX_OBJECTS = None       # None = scan toàn bộ
N_SAMPLE_PATHS = 30
TOP_N_FOLDERS = 100


# Normalize prefix
ROOT_PREFIX = ROOT_PREFIX.strip("/")
if ROOT_PREFIX:
    ROOT_PREFIX += "/"
GCS_CREDENTIALS_PATH

'D:\\University\\Projects\\Individual projects\\Multimodal-Retrieval\\apps\\secrets\\gen-lang-client-0547522732-410672fac05f.json'

## 4. Khởi tạo authenticated GCS client

Cell này:

1. Kiểm tra file credential có tồn tại.
2. Đọc Service Account JSON.
3. Tạo `storage.Client`.
4. Kiểm tra bucket có thể truy cập được.


In [2]:
from google.cloud import storage
from google.oauth2 import service_account
import pandas as pd
from pathlib import PurePosixPath
from pathlib import Path
import re
from IPython.display import display


# ------------------------------------------------------------
# 1. Validate credentials file
# ------------------------------------------------------------
credentials_file = Path(GCS_CREDENTIALS_PATH).expanduser()
if not credentials_file.is_absolute():
    repo_root = globals().get("REPO_ROOT", Path.cwd())
    credentials_file = (Path(repo_root) / credentials_file).resolve()
GCS_CREDENTIALS_PATH = str(credentials_file)

if not credentials_file.exists():
    raise FileNotFoundError(
        f"Không tìm thấy GCS credentials file:\n{GCS_CREDENTIALS_PATH}"
    )


# ------------------------------------------------------------
# 2. Load Service Account credentials
# ------------------------------------------------------------
credentials = service_account.Credentials.from_service_account_file(
    str(credentials_file)
)


# ------------------------------------------------------------
# 3. Create authenticated GCS client
# ------------------------------------------------------------
client = storage.Client(
    credentials=credentials,
    project=credentials.project_id,
)


print("Authenticated successfully")
print("Project ID :", credentials.project_id)
print("Bucket     :", BUCKET_NAME)
print("Prefix     :", ROOT_PREFIX or "<bucket root>")


# ------------------------------------------------------------
# 4. Check bucket access
# ------------------------------------------------------------
bucket = client.bucket(BUCKET_NAME)

try:
    # Chỉ request metadata của bucket để kiểm tra quyền/access
    bucket.reload()
    print("Bucket access: OK")
except Exception as e:
    print("Bucket access: FAILED")
    raise e


Authenticated successfully
Project ID : gen-lang-client-0547522732
Bucket     : aic_ai_2026
Prefix     : <bucket root>
Bucket access: OK


## 5. Quét GCS đệ quy

GCS là object storage. "Folder" thực chất là prefix trong object name.

Cell này liệt kê tất cả blob có `prefix=ROOT_PREFIX`, vì vậy việc scan là đệ quy một cách tự nhiên.

Các cột chính:
- `relative_path`
- `parent_folder`
- `filename`
- `extension`
- `size_bytes`
- `updated`
- `content_type`
- `depth`
- `dataset_level`

In [3]:
def detect_dataset_level(path):
    m = re.search(
        r"(?<![A-Za-z0-9])L(2[1-9]|30)(?![A-Za-z0-9])",
        path,
        flags=re.IGNORECASE,
    )
    return m.group(0).upper() if m else None


def scan_gcs_inventory(bucket_name, prefix="", max_objects=None):
    rows = []

    blobs = client.list_blobs(bucket_name, prefix=prefix)

    kept = 0
    for blob in blobs:
        # Bỏ directory placeholder nếu có.
        if blob.name.endswith("/"):
            continue

        relative_path = (
            blob.name[len(prefix):]
            if prefix and blob.name.startswith(prefix)
            else blob.name
        )

        p = PurePosixPath(relative_path)
        parent = str(p.parent)
        if parent == ".":
            parent = "<root>"

        size_bytes = blob.size or 0

        rows.append({
            "name": blob.name,
            "relative_path": relative_path,
            "parent_folder": parent,
            "filename": p.name,
            "extension": p.suffix.lower() if p.suffix else "<no_ext>",
            "size_bytes": size_bytes,
            "size_mb": size_bytes / (1024 ** 2),
            "updated": blob.updated,
            "content_type": blob.content_type,
            "depth": max(len(p.parts) - 1, 0),
            "dataset_level": detect_dataset_level(relative_path),
        })

        kept += 1

        if kept % 25000 == 0:
            print(f"Scanned {kept:,} objects...")

        if max_objects is not None and kept >= max_objects:
            print(f"Stopped at MAX_OBJECTS={max_objects:,}")
            break

    return pd.DataFrame(rows)


inventory_df = scan_gcs_inventory(
    bucket_name=BUCKET_NAME,
    prefix=ROOT_PREFIX,
    max_objects=MAX_OBJECTS,
)

if inventory_df.empty:
    raise RuntimeError(
        "Không tìm thấy object. Kiểm tra bucket, prefix và quyền GCS."
    )

print(f"Objects scanned: {len(inventory_df):,}")
display(inventory_df.head(20))

Scanned 25,000 objects...
Scanned 50,000 objects...
Scanned 75,000 objects...
Scanned 100,000 objects...
Scanned 125,000 objects...
Scanned 150,000 objects...
Scanned 175,000 objects...
Scanned 200,000 objects...
Scanned 225,000 objects...
Scanned 250,000 objects...
Scanned 275,000 objects...
Scanned 300,000 objects...
Objects scanned: 312,505


,name,relative_path,parent_folder,filename,extension,size_bytes,size_mb,updated,content_type,depth,dataset_level
0,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,_SUCCESS,<no_ext>,0,0.000000,2026-08-02 15:54:03.150000+00:00,text/plain,9,L21
1,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,annotations.jsonl,.jsonl,3655419,3.486079,2026-08-02 15:54:01.980000+00:00,application/x-ndjson,9,L21
2,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,manifest.json,.json,2333,0.002225,2026-08-13 15:46:10.625000+00:00,application/json,9,L21
3,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,latest.json,.json,2740,0.002613,2026-08-13 15:46:09.863000+00:00,application/json,8,L21
4,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,_SUCCESS,<no_ext>,0,0.000000,2026-08-02 10:10:12.437000+00:00,text/plain,10,L21
5,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,annotations.jsonl,.jsonl,61115,0.058284,2026-08-02 10:10:10.250000+00:00,application/x-ndjson,10,L21
6,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,config.json,.json,6340,0.006046,2026-08-02 10:10:11.448000+00:00,application/json,10,L21
7,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,metrics.csv,.csv,1190,0.001135,2026-08-02 10:10:10.536000+00:00,text/csv,10,L21
8,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,run.log,.log,1127,0.001075,2026-08-02 10:10:12.166000+00:00,text/plain,10,L21
9,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,features/extractors/dataset=ai_challenge_2025/...,summary.json,.json,1870,0.001783,2026-08-02 10:10:11.163000+00:00,application/json,10,L21


## 6. Overview

In [4]:
overview_df = pd.DataFrame([{
    "bucket": BUCKET_NAME,
    "root_prefix": ROOT_PREFIX or "<bucket root>",
    "objects": len(inventory_df),
    "total_size_GB": inventory_df["size_bytes"].sum() / (1024 ** 3),
    "unique_parent_folders": inventory_df["parent_folder"].nunique(),
    "max_depth": int(inventory_df["depth"].max()),
    "unique_extensions": inventory_df["extension"].nunique(),
    "detected_levels": inventory_df["dataset_level"].nunique(dropna=True),
    "objects_with_level_detected": int(inventory_df["dataset_level"].notna().sum()),
}])

display(overview_df)

,bucket,root_prefix,objects,total_size_GB,unique_parent_folders,max_depth,unique_extensions,detected_levels,objects_with_level_detected
0,aic_ai_2026,<bucket root>,312505,148.25213,981,10,7,10,312432


## 7. Summary theo extension

In [5]:
extension_summary_df = (
    inventory_df
    .groupby("extension", dropna=False)
    .agg(
        objects=("relative_path", "count"),
        total_size_MB=("size_mb", "sum"),
        avg_size_KB=("size_bytes", lambda x: x.mean() / 1024),
        folders=("parent_folder", "nunique"),
    )
    .reset_index()
    .sort_values(["objects", "total_size_MB"], ascending=[False, False])
)

display(extension_summary_df)

,extension,objects,total_size_MB,avg_size_KB,folders
1,.jpg,310301,58980.071163,194.635508,873
3,.jsonl,990,1099.342652,1137.097855,941
5,.mp4,961,91393.794622,97385.271273,16
2,.json,83,0.537374,6.629777,76
0,.csv,64,335.478252,5367.652039,63
4,.log,62,0.956748,15.801774,62
6,<no_ext>,44,0.000000,0.000000,44


## 8. Summary theo L21–L30

Đây chỉ là object/path inventory, chưa phải caption/OCR/object coverage.

In [6]:
level_df = inventory_df.copy()
level_df["dataset_level"] = level_df["dataset_level"].fillna("<not_detected>")

level_summary_df = (
    level_df
    .groupby("dataset_level")
    .agg(
        objects=("relative_path", "count"),
        total_size_GB=("size_bytes", lambda x: x.sum() / (1024 ** 3)),
        folders=("parent_folder", "nunique"),
    )
    .reset_index()
)

def level_order(value):
    m = re.fullmatch(r"L(\d+)", str(value))
    return int(m.group(1)) if m else 9999

level_summary_df["_order"] = level_summary_df["dataset_level"].map(level_order)
level_summary_df = (
    level_summary_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

display(level_summary_df)

,dataset_level,objects,total_size_GB,folders
0,L21,25748,8.446491,57
1,L22,31301,10.466030,38
2,L23,2593,2.478984,31
3,L24,7291,7.197449,49
4,L25,36530,30.137339,93
5,L26,155261,57.997157,507
6,L27,9013,4.509563,20
7,L28,16067,10.491662,28
8,L29,15141,10.068412,27
9,L30,13487,6.456658,100


## 9. Recursive folder summary

Ví dụ object:

`L26/keyframes/video_001/000123.jpg`

sẽ được tính vào:
- `L26`
- `L26/keyframes`
- `L26/keyframes/video_001`

Nhờ vậy có thể nhìn cấu trúc tree mà không in toàn bộ filename.

In [7]:
def build_recursive_folder_summary(df):
    stats = {}

    for row in df[["relative_path", "size_bytes"]].itertuples(index=False):
        p = PurePosixPath(row.relative_path)
        directories = p.parts[:-1]

        prefix_parts = []
        for depth, directory in enumerate(directories, start=1):
            prefix_parts.append(directory)
            folder = "/".join(prefix_parts)

            if folder not in stats:
                stats[folder] = {
                    "folder": folder,
                    "depth": depth,
                    "objects_recursive": 0,
                    "size_bytes_recursive": 0,
                }

            stats[folder]["objects_recursive"] += 1
            stats[folder]["size_bytes_recursive"] += row.size_bytes

    result = pd.DataFrame(stats.values())

    if result.empty:
        return result

    result["size_GB_recursive"] = (
        result["size_bytes_recursive"] / (1024 ** 3)
    )

    return (
        result
        .sort_values(
            ["depth", "objects_recursive", "folder"],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )


folder_summary_df = build_recursive_folder_summary(inventory_df)
display(folder_summary_df.head(TOP_N_FOLDERS))

,folder,depth,objects_recursive,size_bytes_recursive,size_GB_recursive
0,processed,1,311301,62546581801,58.251044
1,raw,1,961,95833339590,89.251753
2,features,1,173,802439212,0.747330
3,manifests,1,42,1602909,0.001493
4,logs,1,28,548644,0.000511
...,...,...,...,...,...
95,raw/source=kaggle/dataset=ai_challenge_2025/so...,5,176,25698597832,23.933684
96,raw/source=kaggle/dataset=ai_challenge_2025/so...,5,96,4137446126,3.853297
97,features/extractors/dataset=ai_challenge_2025/...,5,63,17454627,0.016256
98,raw/source=kaggle/dataset=ai_challenge_2025/so...,5,43,5796197572,5.398130


## 10. Direct folder summary

Chỉ đếm file có parent trực tiếp là folder đó.

In [8]:
direct_folder_summary_df = (
    inventory_df
    .groupby("parent_folder")
    .agg(
        direct_objects=("relative_path", "count"),
        direct_size_MB=("size_mb", "sum"),
        unique_extensions=("extension", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["direct_objects", "direct_size_MB"],
        ascending=[False, False],
    )
)

display(direct_folder_summary_df.head(TOP_N_FOLDERS))

,parent_folder,direct_objects,direct_size_MB,unique_extensions
115,processed/keyframes/dataset=ai_challenge_2025/...,1300,277.779902,2
103,processed/keyframes/dataset=ai_challenge_2025/...,1249,269.362199,2
117,processed/keyframes/dataset=ai_challenge_2025/...,1195,255.358906,2
112,processed/keyframes/dataset=ai_challenge_2025/...,1138,223.806482,2
100,processed/keyframes/dataset=ai_challenge_2025/...,1135,233.387081,2
...,...,...,...,...
799,processed/keyframes/dataset=ai_challenge_2025/...,604,151.780955,2
794,processed/keyframes/dataset=ai_challenge_2025/...,601,158.455524,2
793,processed/keyframes/dataset=ai_challenge_2025/...,598,125.590857,2
831,processed/keyframes/dataset=ai_challenge_2025/...,595,154.901292,2


## 11. Summary theo depth

In [9]:
depth_summary_df = (
    inventory_df
    .groupby("depth")
    .agg(
        objects=("relative_path", "count"),
        folders=("parent_folder", "nunique"),
        size_GB=("size_bytes", lambda x: x.sum() / (1024 ** 3)),
    )
    .reset_index()
    .sort_values("depth")
)

display(depth_summary_df)

,depth,objects,folders,size_GB
0,3,70,28,0.002004
1,4,2,2,0.000003
2,5,3,3,0.000385
3,6,311288,892,58.293256
4,7,18,5,0.017119
5,8,148,32,0.680674
6,9,3,1,0.003407
7,10,973,18,89.255282


## 12. Keyword audit trong path

Cell này chưa đọc JSON/CSV. Nó chỉ kiểm tra tên object/path có chứa keyword hay không.

In [10]:
KEYWORDS = [
    "caption",
    "ocr",
    "object",
    "keyframe",
    "frame",
    "feature",
    "embedding",
    "metadata",
    "asr",
]

keyword_rows = []
lower_paths = inventory_df["relative_path"].str.lower()

for keyword in KEYWORDS:
    mask = lower_paths.str.contains(keyword, regex=False, na=False)
    matched = inventory_df.loc[mask]

    keyword_rows.append({
        "keyword": keyword,
        "matched_objects": len(matched),
        "matched_folders": matched["parent_folder"].nunique(),
        "size_GB": matched["size_bytes"].sum() / (1024 ** 3),
        "sample_path": (
            matched["relative_path"].iloc[0]
            if len(matched) > 0
            else None
        ),
    })

keyword_summary_df = pd.DataFrame(keyword_rows)
display(keyword_summary_df)

,keyword,matched_objects,matched_folders,size_GB,sample_path
0,caption,19,5,0.006944,features/extractors/dataset=ai_challenge_2025/...
1,ocr,29,7,0.078500,features/extractors/dataset=ai_challenge_2025/...
2,object,120,26,0.602173,features/extractors/dataset=ai_challenge_2025/...
3,keyframe,311298,894,58.250662,processed/keyframes/dataset=ai_challenge_2025/...
4,frame,311471,934,58.997992,features/extractors/dataset=ai_challenge_2025/...
5,feature,176,43,0.747711,features/extractors/dataset=ai_challenge_2025/...
6,embedding,3,3,0.000381,processed/feature-annotations/fe-vector-embedd...
7,metadata,5,1,0.000007,features/import_bundles/dataset=ai_challenge_2...
8,asr,0,0,0.000000,NaN


## 13. Sample path theo extension

In [11]:
rows = []

for extension, group in inventory_df.groupby("extension"):
    for path in group["relative_path"].head(10):
        rows.append({
            "extension": extension,
            "sample_path": path,
        })

sample_by_extension_df = pd.DataFrame(rows)
display(sample_by_extension_df.head(200))

,extension,sample_path
0,.csv,features/extractors/dataset=ai_challenge_2025/...
1,.csv,features/extractors/dataset=ai_challenge_2025/...
2,.csv,features/extractors/dataset=ai_challenge_2025/...
3,.csv,features/extractors/dataset=ai_challenge_2025/...
4,.csv,features/extractors/dataset=ai_challenge_2025/...
...,...,...
65,<no_ext>,features/extractors/dataset=ai_challenge_2025/...
66,<no_ext>,features/extractors/dataset=ai_challenge_2025/...
67,<no_ext>,features/extractors/dataset=ai_challenge_2025/...
68,<no_ext>,features/extractors/dataset=ai_challenge_2025/...


## 14. Sample path theo L21–L30

Đây là một trong các output quan trọng nhất cần gửi lại để viết parser chính xác.

In [12]:
tmp = inventory_df.copy()
tmp["dataset_level"] = tmp["dataset_level"].fillna("<not_detected>")

rows = []

for level, group in tmp.groupby("dataset_level"):
    for path in group["relative_path"].head(N_SAMPLE_PATHS):
        rows.append({
            "dataset_level": level,
            "sample_path": path,
        })

sample_by_level_df = pd.DataFrame(rows)
display(sample_by_level_df.head(400))

,dataset_level,sample_path
0,<not_detected>,logs/pipeline=kaggle_ingest/run_id=20260701T11...
1,<not_detected>,logs/pipeline=kaggle_ingest/run_id=20260701T11...
2,<not_detected>,logs/pipeline=kaggle_ingest/run_id=20260702T13...
3,<not_detected>,logs/pipeline=kaggle_ingest/run_id=20260702T13...
4,<not_detected>,logs/pipeline=kaggle_ingest/run_id=20260702T14...
...,...,...
325,L30,processed/keyframes/dataset=ai_challenge_2025/...
326,L30,processed/keyframes/dataset=ai_challenge_2025/...
327,L30,processed/keyframes/dataset=ai_challenge_2025/...
328,L30,processed/keyframes/dataset=ai_challenge_2025/...


## 15. Xem naming convention của filename

In [ ]:
filename_samples_df = (
    inventory_df[
        [
            "dataset_level",
            "parent_folder",
            "filename",
            "extension",
            "size_bytes",
        ]
    ]
    .sort_values(
        ["dataset_level", "parent_folder", "filename"],
        na_position="last",
    )
    .head(500)
)

display(filename_samples_df)

## 16. Export CSV + Excel

Raw inventory được giữ riêng trong CSV.

Excel chứa các sheet summary và một sheet inventory. Với dataset dưới khoảng 1 triệu objects có thể mở bằng Excel; CSV vẫn là source phù hợp hơn cho raw inventory.

In [13]:
import os

OUTPUT_DIR = "gcs_explorer_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

inventory_path = os.path.join(OUTPUT_DIR, "gcs_inventory.csv")
folder_path = os.path.join(OUTPUT_DIR, "gcs_folder_summary.csv")
direct_folder_path = os.path.join(
    OUTPUT_DIR,
    "gcs_direct_folder_summary.csv",
)
extension_path = os.path.join(
    OUTPUT_DIR,
    "gcs_extension_summary.csv",
)
level_path = os.path.join(OUTPUT_DIR, "gcs_level_summary.csv")
keyword_path = os.path.join(OUTPUT_DIR, "gcs_keyword_summary.csv")
sample_level_path = os.path.join(
    OUTPUT_DIR,
    "gcs_sample_paths_by_level.csv",
)
excel_path = os.path.join(OUTPUT_DIR, "gcs_explorer_report.xlsx")

inventory_df.to_csv(inventory_path, index=False)
folder_summary_df.to_csv(folder_path, index=False)
direct_folder_summary_df.to_csv(direct_folder_path, index=False)
extension_summary_df.to_csv(extension_path, index=False)
level_summary_df.to_csv(level_path, index=False)
keyword_summary_df.to_csv(keyword_path, index=False)
sample_by_level_df.to_csv(sample_level_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    overview_df.to_excel(writer, sheet_name="overview", index=False)
    level_summary_df.to_excel(writer, sheet_name="levels", index=False)
    extension_summary_df.to_excel(
        writer,
        sheet_name="extensions",
        index=False,
    )
    depth_summary_df.to_excel(writer, sheet_name="depth", index=False)
    keyword_summary_df.to_excel(
        writer,
        sheet_name="keywords",
        index=False,
    )
    folder_summary_df.to_excel(
        writer,
        sheet_name="folders_recursive",
        index=False,
    )
    direct_folder_summary_df.to_excel(
        writer,
        sheet_name="folders_direct",
        index=False,
    )
    sample_by_extension_df.to_excel(
        writer,
        sheet_name="samples_extension",
        index=False,
    )
    sample_by_level_df.to_excel(
        writer,
        sheet_name="samples_level",
        index=False,
    )

    # Excel có giới hạn khoảng 1,048,576 rows/sheet.
    inventory_df.head(1_048_000).to_excel(
        writer,
        sheet_name="inventory",
        index=False,
    )

print("Export complete:")
print("-", inventory_path)
print("-", folder_path)
print("-", direct_folder_path)
print("-", extension_path)
print("-", level_path)
print("-", keyword_path)
print("-", sample_level_path)
print("-", excel_path)

ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.

## 17. Output cần gửi lại

Sau khi chạy xong, gửi lại file:

`gcs_explorer_output/gcs_explorer_report.xlsx`

hoặc ít nhất output của:

1. `overview_df`
2. `level_summary_df`
3. `extension_summary_df`
4. `keyword_summary_df`
5. `folder_summary_df.head(100)`
6. `sample_by_level_df`

Sau đó có thể viết notebook bước 2 để kiểm soát chính xác:

- videos
- frames/keyframes
- caption coverage
- OCR coverage
- object coverage
- missing rows
- missing videos
- completion %
- GCS ↔ Elasticsearch mismatch
- dashboard tổng hợp L21–L30

In [ ]:
print("=== OVERVIEW ===")
display(overview_df)

print("\n=== LEVEL SUMMARY ===")
display(level_summary_df)

print("\n=== EXTENSION SUMMARY ===")
display(extension_summary_df.head(50))

print("\n=== KEYWORD SUMMARY ===")
display(keyword_summary_df)

print("\n=== TOP RECURSIVE FOLDERS ===")
display(folder_summary_df.head(100))

print("\n=== SAMPLE PATHS BY LEVEL ===")
display(sample_by_level_df.head(300))